<a href="https://colab.research.google.com/github/johnkarigi/AVIATION-ACCIDENT-ANALYSIS/blob/main/Website_AB_Testing_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Website A/B Testing - Lab

## Introduction

In this lab, you'll get another chance to practice your skills at conducting a full A/B test analysis. It will also be a chance to practice your data exploration and processing skills! The scenario you'll be investigating is data collected from the homepage of a music app page for audacity.

## Objectives

You will be able to:
* Analyze the data from a website A/B test to draw relevant conclusions
* Explore and analyze web action data

## Exploratory Analysis

Start by loading in the dataset stored in the file 'homepage_actions.csv'. Then conduct an exploratory analysis to get familiar with the data.

> Hints:
    * Start investigating the id column:
        * How many viewers also clicked?
        * Are there any anomalies with the data; did anyone click who didn't view?
        * Is there any overlap between the control and experiment groups?
            * If so, how do you plan to account for this in your experimental design?

In [7]:
from google.colab import files
uploaded =  files.upload()

Saving homepage_actions.csv to homepage_actions.csv


In [9]:
#Your code here
import pandas as pd

df = pd.read_csv("homepage_actions.csv")
df.head()


,timestamp,id,group,action
0,2016-09-24 17:42:27.839496,804196,experiment,view
1,2016-09-24 19:19:03.542569,434745,experiment,view
2,2016-09-24 19:36:00.944135,507599,experiment,view
3,2016-09-24 19:59:02.646620,671993,control,view
4,2016-09-24 20:26:14.466886,536734,experiment,view


In [10]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8188 entries, 0 to 8187
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  8188 non-null   object
 1   id         8188 non-null   int64 
 2   group      8188 non-null   object
 3   action     8188 non-null   object
dtypes: int64(1), object(3)
memory usage: 256.0+ KB


,id
count,8188.000000
mean,564699.749878
std,219085.845672
min,182988.000000
25%,373637.500000
50%,566840.500000
75%,758078.000000
max,937217.000000


## Conduct a Statistical Test

Conduct a statistical test to determine whether the experimental homepage was more effective than that of the control group.

In [13]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
# create binary indicators
df['viewed'] = (df['action'] == 'view').astype(int)
df['clicked'] = (df['action'] == 'click').astype(int)


In [14]:
users = df.groupby(['id','group']).agg(
    viewed=('viewed','max'),
    clicked=('clicked','max')
).reset_index()

In [15]:
users = users[users['viewed'] == 1]

In [16]:
# clicks and totals
clicks = users.groupby('group')['clicked'].sum()
totals = users.groupby('group')['clicked'].count()

# two-proportion z-test (experiment > control)
z_stat, p_value = proportions_ztest(
    count=clicks.values,
    nobs=totals.values,
    alternative='larger'
)

z_stat, p_value


(np.float64(-2.618563885349469), np.float64(0.9955849622117021))

## Verifying Results

One sensible formulation of the data to answer the hypothesis test above would be to create a binary variable representing each individual in the experiment and control group. This binary variable would represent whether or not that individual clicked on the homepage; 1 for they did and 0 if they did not.

The variance for the number of successes in a sample of a binomial variable with n observations is given by:

## $n\bullet p (1-p)$

Given this, perform 3 steps to verify the results of your statistical test:
1. Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group.
2. Calculate the number of standard deviations that the actual number of clicks was from this estimate.
3. Finally, calculate a p-value using the normal distribution based on this z-score.

### Step 1:
Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group.

In [18]:
#Your code here
# counts for control and experiment
control_users = users[users['group'] == 'control']['id'].nunique()
control_clicks = users[(users['group'] == 'control') & (users['clicked'] == 1)]['id'].nunique()

experiment_users = users[users['group'] == 'experiment']['id'].nunique()

# control click-through rate
p_control = control_clicks / control_users

# expected clicks in experiment if it behaved like control
expected_clicks_experiment = experiment_users * p_control

control_users, control_clicks, experiment_users, expected_clicks_experiment



(3332, 932, 2996, 838.0168067226891)

### Step 2:
Calculate the number of standard deviations that the actual number of clicks was from this estimate.

In [19]:
#Your code here
# observed clicks in experiment group
experiment_clicks = users[
    (users['group'] == 'experiment') & (users['clicked'] == 1)
]['id'].nunique()

# standard deviation under control rate
std_dev = (experiment_users * p_control * (1 - p_control)) ** 0.5

# number of standard deviations from expected (z-score)
z_score = (experiment_clicks - expected_clicks_experiment) / std_dev

experiment_clicks, std_dev, z_score


(928, 24.568547907005815, 3.6625360854823588)

### Step 3:
Finally, calculate a p-value using the normal distribution based on this z-score.

In [21]:
import pandas as pd
import numpy as np
from scipy.stats import norm

In [22]:
#Your code here
# Create a binary click indicator per user
clicks_per_user = df.groupby(['id', 'group'])['action'].apply(lambda x: 1 if 'click' in x.values else 0).reset_index()
clicks_per_user.rename(columns={'action':'clicked'}, inplace=True)

# Split groups
control = clicks_per_user[clicks_per_user['group'] == 'control']
experiment = clicks_per_user[clicks_per_user['group'] == 'experiment']

# Step 1: Expected clicks in experiment group if it had control's click-through rate
control_click_rate = control['clicked'].mean()
experiment_size = len(experiment)
expected_clicks_experiment = experiment_size * control_click_rate

# Step 2: Standard deviation of expected clicks (binomial variance)
std_clicks_experiment = np.sqrt(experiment_size * control_click_rate * (1 - control_click_rate))

# Step 3: Actual clicks in experiment group
actual_clicks_experiment = experiment['clicked'].sum()

# Step 4: z-score and p-value
z_score = (actual_clicks_experiment - expected_clicks_experiment) / std_clicks_experiment
p_value = 1 - norm.cdf(z_score)  # one-sided test: is experiment better?

expected_clicks_experiment, std_clicks_experiment, actual_clicks_experiment,

(np.float64(838.0168067226891), np.float64(24.568547907005815), np.int64(928))

### Analysis:

Does this result roughly match that of the previous statistical test?

> Comment: **Your analysis here**

## Summary

In this lab, you continued to get more practice designing and conducting AB tests. This required additional work preprocessing and formulating the initial problem in a suitable manner. Additionally, you also saw how to verify results, strengthening your knowledge of binomial variables, and reviewing initial statistical concepts of the central limit theorem, standard deviation, z-scores, and their accompanying p-values.